# Phase 4 — NLP Feature Extraction
## Notebook 04.02 — TF-IDF Feature Extraction

### Goal
Build and compare reproducible **smoke-test** TF-IDF pipelines for the two Phase 3 text representations.

### Methods
- Word TF-IDF with unigrams and bigrams
- Character TF-IDF
- Word + Character TF-IDF

### Leakage rule
The vectorizers in this notebook are fit only on a fixed smoke sample to verify code, memory behavior, and output contracts. They are **not final model features**. After the temporal split, the final vectorizer must be fit on the training split only.

## 1. Imports and repository paths

### Goal
Load the current project without using machine-specific absolute paths.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_repository_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "1_data_acquisition").exists()
            and (candidate / "3_text_preprocessing").exists()
            and (candidate / "4_nlp_feature_extraction").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


PROJECT_ROOT = locate_repository_root()
PHASE4_DIR = PROJECT_ROOT / "4_nlp_feature_extraction"
if str(PHASE4_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE4_DIR))

from src.tfidf_features import (
    load_feature_recipes,
    make_smoke_sample,
    normalize_texts,
    run_tfidf_smoke_benchmark,
    save_tfidf_smoke_outputs,
)

print("Project root:", PROJECT_ROOT)
print("Phase 4 directory:", PHASE4_DIR)

Project root: C:\Users\sepehr\PycharmProjects\FinancialNLP
Phase 4 directory: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction


### Result
The notebook uses repository-relative paths.

### Interpretation
This keeps the workflow portable across machines.

### Decision
Keep the notebook as the educational interface and reusable TF-IDF logic in `src/tfidf_features.py`.

## 2. Review Stage 1 findings before vectorization

### Goal
Carry forward the real missing/empty-text findings rather than treating both representations as equally complete.

In [2]:
STAGE1_COMPARISON_PATH = PHASE4_DIR / "data" / "reports" / "representation_comparison.csv"
stage1_comparison = pd.read_csv(STAGE1_COMPARISON_PATH)
display(stage1_comparison)

,representation,row_count,null_count,null_pct,empty_count,empty_pct,nonempty_count,nonempty_pct,median_word_count,p95_word_count,max_word_count,median_character_count,p95_character_count,max_character_count,very_short_word_count,very_short_word_pct_nonempty,very_short_character_count,very_short_character_pct_nonempty
0,text_title_description,88936,0,0.0000,76,0.0855,88860,99.9145,31.0000,72.0000,958,199.0000,456.0000,5705,0,0.0000,0,0.0000
1,Filtered_Text,88936,1,0.0011,1,0.0011,88935,99.9989,60.0000,318.0000,1636,454.0000,"2,432.0000",12724,530,0.5959,468,0.5262


### Result
The current Stage 1 report contains 88,936 rows. It records 76 empty `text_title_description` rows and one null/empty `Filtered_Text` row.

### Interpretation
An empty document normally becomes a TF-IDF zero-vector. This is valid sparse output, but it must be counted and reported.

`Filtered_Text` is longer on average and has a much larger 95th-percentile length. It may create more vocabulary and character n-grams, increasing runtime and memory.

### Decision
Do not delete empty rows. Force the smoke sample to include them, preserve `source_row_id`, and report zero-vectors separately for every experiment.

## 3. Load TF-IDF recipes

### Goal
Keep vocabulary limits, n-gram ranges, and the fixed smoke-sample seed in one readable YAML file.

In [3]:
CONFIG_PATH = PHASE4_DIR / "configs" / "feature_recipes.yaml"
config = load_feature_recipes(CONFIG_PATH)

print("Config:", CONFIG_PATH)
print("Representations:", config["input"]["representations"])
print("Smoke sample size:", config["smoke_test"]["sample_size"])
print("Random state:", config["smoke_test"]["random_state"])
print("Recipes:", list(config["recipes"]))

Config: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\configs\feature_recipes.yaml
Representations: ['text_title_description', 'Filtered_Text']
Smoke sample size: 3000
Random state: 42
Recipes: ['word_tfidf', 'character_tfidf', 'word_character_tfidf']


### Interpretation
`max_features` keeps the smoke benchmark bounded. These are engineering limits for a quick comparison, not final hyperparameters.

### Decision
Tune final settings only after the temporal split and baseline evaluation are available.

## 4. Read the current Phase 3 dataset

### Goal
Load only the identifier and two requested representations.

In [4]:
INPUT_PATH = PROJECT_ROOT / config["input"]["relative_path"]
ID_COLUMN = config["input"]["id_column"]
REPRESENTATIONS = config["input"]["representations"]

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Phase 3 parquet was not found: {INPUT_PATH}\nRun Phase 3 first."
    )

text_df = pd.read_parquet(INPUT_PATH, columns=[ID_COLUMN, *REPRESENTATIONS])
print("Input shape:", text_df.shape)
print("ID nulls:", int(text_df[ID_COLUMN].isna().sum()))
print("ID duplicates:", int(text_df[ID_COLUMN].duplicated().sum()))
text_df.head()

Input shape: (88936, 3)
ID nulls: 0
ID duplicates: 0


,source_row_id,text_title_description,Filtered_Text
0,0,"Anonymous Email Service ProtonMail Adds Bitcoin Payment Option ProtonMail has revealed it is adding bitcoin as an official, automated pa...",protonmail revealed adding official automated payment option cryptocurrencies may soon follow
1,1,Blockchain & Bitcoin Conference Stockholm to Feature Discussions on ICOs and Blockchain Development,september stockholm host large conference dedicated blockchain technology cryptocurrencies icos blockchain conference stockholm julian p...
2,2,"Bitcoin Prices Reach New All-Time High of Over $4,500 Following two days of sideways trading, bitcoin prices have once again reached rec...",following two days sideways trading prices reached record levels
3,3,Lightning Bank Ledgers? Bitfury and Ripple Demo New Twist on Bitcoin Tech Bitcoin's much-anticipated Lightning Network is now compatible...,much-anticipated lightning network compatible seven payment networks thanks new code ripple bitfury
4,4,$7 Million: Bitcoin Wallet Startup Breadwallet Raises New Funding Bitcoin wallet startup Breadwallet has closed a new $7 million funding...,wallet startup breadwallet closed new million funding boost staffing levels product development


### Decision
Stop only if the input file or required columns are unavailable. Empty text is not a structural error.

## 5. Build a fixed smoke sample

### Goal
Create a reproducible sample that also includes all rows empty in either representation.

In [5]:
smoke_cfg = config["smoke_test"]
smoke_df = make_smoke_sample(
    text_df,
    id_column=ID_COLUMN,
    representations=REPRESENTATIONS,
    sample_size=smoke_cfg["sample_size"],
    random_state=smoke_cfg["random_state"],
    include_all_empty_rows=smoke_cfg["include_all_empty_rows"],
)

sample_summary = pd.DataFrame({
    "representation": REPRESENTATIONS,
    "sample_rows": [len(smoke_df)] * len(REPRESENTATIONS),
    "empty_rows": [
        int(normalize_texts(smoke_df[column]).eq("").sum())
        for column in REPRESENTATIONS
    ],
})
print("Sample source_row_id order preserved:", smoke_df[ID_COLUMN].is_monotonic_increasing)
display(sample_summary)

Sample source_row_id order preserved: True


,representation,sample_rows,empty_rows
0,text_title_description,3000,76
1,Filtered_Text,3000,1


### Interpretation
The sample is stable for the same dataset and seed. Empty rows are deliberately retained, so zero-vector reporting is exercised every time.

### Decision
Use this sample only for smoke testing. It is not a train/test split and must not be used to report predictive performance.

## 6. Understand the three TF-IDF methods

### Word TF-IDF
Uses word tokens. Unigrams capture individual terms such as `bitcoin` or `approval`; bigrams capture short phrases such as `rate cut` or `spot etf`.

**Strengths:** interpretable vocabulary and strong classical baseline.  
**Weaknesses:** unseen words and spelling variants are not represented directly.

### Character TF-IDF
Uses character n-grams inside word boundaries.

**Strengths:** robust to spelling variation, ticker forms, morphology, and rare terms.  
**Weaknesses:** often produces more features and is less directly interpretable.

### Word + Character TF-IDF
Horizontally combines the two sparse matrices.

**Strengths:** preserves semantic word features and subword robustness.  
**Weaknesses:** largest feature space and memory cost.

All three outputs remain sparse CSR matrices.

## 7. Run all six smoke-test experiments

### Goal
Fit three recipes independently on both text representations and record technical diagnostics.

In [6]:
smoke_result = run_tfidf_smoke_benchmark(
    smoke_df,
    id_column=ID_COLUMN,
    representations=REPRESENTATIONS,
    recipes=config["recipes"],
)

benchmark = smoke_result["benchmark"].sort_values(
    ["representation", "recipe"], kind="stable"
).reset_index(drop=True)
display(benchmark)

,representation,recipe,empty_input_rows,source_row_id_order_preserved,rows,features,shape,nonzero_values,sparsity_pct,zero_vectors,zero_vector_pct,elapsed_seconds,approx_sparse_memory_mb,is_sparse_csr,finite_values
0,Filtered_Text,character_tfidf,1,True,3000,6000,"(3000, 6000)",2503747,86.0903,1,0.0333,5.6186,19.1135,True,True
1,Filtered_Text,word_character_tfidf,1,True,3000,12000,"(3000, 12000)",2774582,92.2928,1,0.0333,7.8352,21.1798,True,True
2,Filtered_Text,word_tfidf,1,True,3000,6000,"(3000, 6000)",270835,98.4954,2,0.0667,1.9283,2.0778,True,True
3,text_title_description,character_tfidf,76,True,3000,6000,"(3000, 6000)",771911,95.7116,76,2.5333,0.9251,5.9007,True,True
4,text_title_description,word_character_tfidf,76,True,3000,12000,"(3000, 12000)",866535,97.5930,76,2.5333,2.1167,6.6226,True,True
5,text_title_description,word_tfidf,76,True,3000,6000,"(3000, 6000)",94624,99.4743,76,2.5333,0.1955,0.7334,True,True


### Result
The benchmark reports:
- matrix shape and number of features;
- nonzero values and sparsity;
- zero-vector count and percentage;
- fit/transform time;
- approximate CSR memory;
- sparse-format and finite-value checks;
- `source_row_id` order preservation.

### Interpretation
A higher feature count is not automatically better. The useful comparison is later predictive performance under the same temporal split, model, and evaluation protocol.

## 8. Inspect zero-vectors and sparse safety

### Goal
Verify that empty or vocabulary-free documents are visible in the report and that no dense conversion occurred.

In [7]:
zero_vector_report = benchmark[[
    "representation",
    "recipe",
    "empty_input_rows",
    "zero_vectors",
    "zero_vector_pct",
    "is_sparse_csr",
    "finite_values",
    "source_row_id_order_preserved",
]]
display(zero_vector_report)

assert benchmark["is_sparse_csr"].all()
assert benchmark["finite_values"].all()
assert benchmark["source_row_id_order_preserved"].all()
assert (benchmark["rows"] == len(smoke_df)).all()

,representation,recipe,empty_input_rows,zero_vectors,zero_vector_pct,is_sparse_csr,finite_values,source_row_id_order_preserved
0,Filtered_Text,character_tfidf,1,1,0.0333,True,True,True
1,Filtered_Text,word_character_tfidf,1,1,0.0333,True,True,True
2,Filtered_Text,word_tfidf,1,2,0.0667,True,True,True
3,text_title_description,character_tfidf,76,76,2.5333,True,True,True
4,text_title_description,word_character_tfidf,76,76,2.5333,True,True,True
5,text_title_description,word_tfidf,76,76,2.5333,True,True,True


### Interpretation
Zero-vectors can exceed the number of empty inputs when a non-empty text contains no token or n-gram surviving `min_df`, `max_df`, and tokenization rules.

### Decision
Report zero-vectors instead of silently removing rows. Later models should preserve these rows and let the evaluation reveal whether a fallback feature is needed.

## 9. Save sparse smoke artifacts

### Goal
Save `.npz` matrices without converting them to dense form and preserve a separate ordered ID mapping.

In [8]:
OUTPUT_DIR = PROJECT_ROOT / config["output"]["directory_relative_path"]
BENCHMARK_PATH = PROJECT_ROOT / config["output"]["benchmark_relative_path"]
METADATA_PATH = PROJECT_ROOT / config["output"]["metadata_relative_path"]

saved_files = save_tfidf_smoke_outputs(
    smoke_result,
    output_directory=OUTPUT_DIR,
    benchmark_path=BENCHMARK_PATH,
    metadata_path=METADATA_PATH,
    config=config,
)

print("Saved files:")
for name, path in saved_files.items():
    print(f"- {name}: {path}")

Saved files:
- source_row_ids: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\tfidf_smoke\smoke_sample_source_row_ids.csv
- matrix::text_title_description__word_tfidf: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\tfidf_smoke\text_title_description__word_tfidf.npz
- features::text_title_description__word_tfidf: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\tfidf_smoke\text_title_description__word_tfidf_feature_names.json
- vectorizers::text_title_description__word_tfidf: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\tfidf_smoke\text_title_description__word_tfidf_vectorizers.joblib
- matrix::text_title_description__character_tfidf: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_extraction\data\tfidf_smoke\text_title_description__character_tfidf.npz
- features::text_title_description__character_tfidf: C:\Users\sepehr\PycharmProjects\FinancialNLP\4_nlp_feature_ext

### Result
Each experiment receives a sparse `.npz` matrix, feature-name JSON, and fitted smoke vectorizer file. One shared CSV stores `source_row_id` in exact matrix row order.

### Decision
Generated matrices are ignored by Git because they are reproducible artifacts. Reports and code remain small and reviewable.

## 10. Leakage-safe handoff

### Final decision
Do **not** fit final TF-IDF on all 88,936 rows.

After Phase 6 defines the temporal split:

```text
train text -> fit vectorizer
train text -> transform
validation text -> transform only
 test text -> transform only
```

If TruncatedSVD is used later, fit it on the training TF-IDF matrix only and apply the fitted transformer to validation/test matrices.

Stage 2 is complete when all six smoke experiments run, remain sparse and finite, preserve `source_row_id` order, and write reproducible diagnostic artifacts.